##### Mole Classifier Kernel
Skin cancer is the most common human malignancy, is primarily diagnosed visually, beginning with an initial clinical screening and followed potentially by dermoscopic analysis, a biopsy and histopathological examination. Automated classification of skin lesions using images is a challenging task owing to the fine-grained variability in the appearance of skin lesions.

The dataset is taken from the ISIC (International Skin Image Collaboration) Archive. It consists of 1800 pictures of benign moles and 1497 pictures of malignant classified moles. The pictures have all been resized to low resolution (224x224x3) RGB. The task of this kernel is to create a model, which can classify a mole visually into benign and malignant. 

As the dataset is pretty balanced, the model will be tested on the accuracy score, thus (TP + TN)/(ALL).

It has 2 different classes of skin cancer which are listed below :<br>
**1. Benign <br>**
**2. Malignant <br>**

In this kernel I will try to detect 2 different classes of moles using Convolution Neural Network with keras tensorflow in backend and then analyse the result to see how the model can be useful in practical scenario.<br>

In this kernel I have followed following 14 steps for model building and evaluation which are as follows : <br>
**Step 1 : Importing Essential Libraries**<br>
**Step 2: Loading pictures and making Dictionary of images and labels** <br>
**Step 3: Categorical Labels** <br>
**Step 4: Normalization** <br>
**Step 5: Train and Test Split** <br>
**Step 6: Model Building **<br>
**Step 7: Cross-validating model**<br>
**Step 8: Testing model**<br>
**Step 9: ResNet50** <br>

# Step 1 : importing Essential Libraries

In [ ]:
import os
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from glob import glob
import seaborn as sns
from PIL import Image
np.random.seed(11)

from sklearn.preprocessing import StandardScaler 
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import itertools

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.resnet50 import ResNet50

print(f'TensorFlow Version: {tf.__version__}')


# Step 2 : Loading pictures and making Dictionary of images and labels
In this step I load in the pictures and turn them into numpy arrays using their RGB values. As the pictures have already been resized to 224x224, there's no need to resize them. As the pictures do not have any labels, these need to be created. Finally, the pictures are added together to a big training set and shuffeled.

In [ ]:
# ====================================================================
# 🌐 DATASET AUTO-RESOLVER & KAGGLE CONNECTOR (Skin Cancer Detection)
# ====================================================================
import os
from PIL import Image
import numpy as np

# Candidate locations in Kaggle or local
candidate_roots = [
    '/kaggle/input/skin-cancer-malignant-vs-benign',
    '/kaggle/input/data',
    '../input/data',
    '../input/skin-cancer-malignant-vs-benign',
    '/kaggle/input',
    '../input',
    '../datasets/skin_cancer',
    './datasets/skin_cancer',
    '.'
]

train_dir, test_dir = None, None

def find_benign_malignant(base):
    tr, ts = None, None
    if not os.path.exists(base):
        return tr, ts
    for root, dirs, _ in os.walk(base):
        lower_dirs = [d.lower() for d in dirs]
        if 'benign' in lower_dirs and 'malignant' in lower_dirs:
            if 'train' in root.lower():
                tr = root
            elif 'test' in root.lower() or 'val' in root.lower():
                ts = root
            elif tr is None:
                tr = root
    return tr, ts

for c in candidate_roots:
    tr, ts = find_benign_malignant(c)
    if tr:
        train_dir = tr
        test_dir = ts if ts else tr
        print(f'✅ Found dataset at: {c}')
        print(f'📂 Train folder: {train_dir}')
        print(f'📂 Test folder:  {test_dir}')
        break

if not train_dir:
    print('⚡ Attempting automatic KaggleHub download...')
    try:
        import kagglehub
        dl_path = kagglehub.dataset_download('fanconic/skin-cancer-malignant-vs-benign')
        print(f'✅ Downloaded to: {dl_path}')
        tr, ts = find_benign_malignant(dl_path)
        train_dir, test_dir = tr, ts if ts else tr
    except Exception as e:
        print(f'⚠️ KaggleHub notice: {e}')

if not train_dir or not os.path.exists(train_dir):
    raise FileNotFoundError(
        'Skin Cancer dataset not found! Please attach "fanconic/skin-cancer-malignant-vs-benign" to your Kaggle notebook.'
    )

folder_benign_train = os.path.join(train_dir, [d for d in os.listdir(train_dir) if d.lower() == 'benign'][0])
folder_malignant_train = os.path.join(train_dir, [d for d in os.listdir(train_dir) if d.lower() == 'malignant'][0])

folder_benign_test = os.path.join(test_dir, [d for d in os.listdir(test_dir) if d.lower() == 'benign'][0])
folder_malignant_test = os.path.join(test_dir, [d for d in os.listdir(test_dir) if d.lower() == 'malignant'][0])

read = lambda imname: np.asarray(Image.open(imname).convert('RGB').resize((224, 224)))

print('Loading training pictures...')
ims_benign = [read(os.path.join(folder_benign_train, f)) for f in os.listdir(folder_benign_train) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('.')]
X_benign = np.array(ims_benign, dtype='uint8')
ims_malignant = [read(os.path.join(folder_malignant_train, f)) for f in os.listdir(folder_malignant_train) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('.')]
X_malignant = np.array(ims_malignant, dtype='uint8')

print('Loading testing pictures...')
ims_benign = [read(os.path.join(folder_benign_test, f)) for f in os.listdir(folder_benign_test) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('.')]
X_benign_test = np.array(ims_benign, dtype='uint8')
ims_malignant = [read(os.path.join(folder_malignant_test, f)) for f in os.listdir(folder_malignant_test) if f.lower().endswith(('.jpg', '.jpeg', '.png')) and not f.startswith('.')]
X_malignant_test = np.array(ims_malignant, dtype='uint8')

# Labels: 0 = Benign, 1 = Malignant
y_benign = np.zeros(X_benign.shape[0])
y_malignant = np.ones(X_malignant.shape[0])
y_benign_test = np.zeros(X_benign_test.shape[0])
y_malignant_test = np.ones(X_malignant_test.shape[0])

# Merge
X_train = np.concatenate((X_benign, X_malignant), axis=0)
y_train = np.concatenate((y_benign, y_malignant), axis=0)
X_test = np.concatenate((X_benign_test, X_malignant_test), axis=0)
y_test = np.concatenate((y_benign_test, y_malignant_test), axis=0)

# Shuffle
s = np.arange(X_train.shape[0])
np.random.shuffle(s)
X_train, y_train = X_train[s], y_train[s]

s = np.arange(X_test.shape[0])
np.random.shuffle(s)
X_test, y_test = X_test[s], y_test[s]

print(f'✅ Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Classes: 0: Benign ({len(y_benign)} train, {len(y_benign_test)} test) | 1: Malignant ({len(y_malignant)} train, {len(y_malignant_test)} test)')


In [ ]:
# Display first 15 images of moles, and how they are classified
w=40
h=30
fig=plt.figure(figsize=(12, 8))
columns = 5
rows = 3

for i in range(1, columns*rows +1):
    ax = fig.add_subplot(rows, columns, i)
    if y_train[i] == 0:
        ax.title.set_text('Benign')
    else:
        ax.title.set_text('Malignant')
    plt.imshow(X_train[i], interpolation='nearest')
plt.show()

# Step 3: Categorical Labels
Turn labels into one hot encoding

In [ ]:
y_train = to_categorical(y_train, num_classes= 2)
y_test = to_categorical(y_test, num_classes= 2)

# Step 4 : Normalization
Normalize all Values of the pictures by dividing all the RGB values by 255

In [ ]:
# With data augmentation to prevent overfitting 
X_train = X_train/255.
X_test = X_test/255.

# Step 5: Model Building 
## CNN
I used the Keras Sequential API, where you have just to add one layer at a time, starting from the input.

The first is the convolutional (Conv2D) layer. It is like a set of learnable filters. I choosed to set 64 filters for the two firsts conv2D layers. Each filter transforms a part of the image (defined by the kernel size) using the kernel filter. The kernel filter matrix is applied on the whole image. Filters can be seen as a transformation of the image.

The CNN can isolate features that are useful everywhere from these transformed images (feature maps).

The second important layer in CNN is the pooling (MaxPool2D) layer. This layer simply acts as a downsampling filter. It looks at the 2 neighboring pixels and picks the maximal value. These are used to reduce computational cost, and to some extent also reduce overfitting. We have to choose the pooling size (i.e the area size pooled each time) more the pooling dimension is high, more the downsampling is important.

Combining convolutional and pooling layers, CNN are able to combine local features and learn more global features of the image.

Dropout is a regularization method, where a proportion of nodes in the layer are randomly ignored (setting their wieghts to zero) for each training sample. This drops randomly a propotion of the network and forces the network to learn features in a distributed way. This technique also improves generalization and reduces the overfitting.

'relu' is the rectifier (activation function max(0,x). The rectifier activation function is used to add non linearity to the network.

The Flatten layer is use to convert the final feature maps into a one single 1D vector. This flattening step is needed so that you can make use of fully connected layers after some convolutional/maxpool layers. It combines all the found local features of the previous convolutional layers.

In the end i used the features in one fully-connected (Dense) layer which is just artificial an neural networks (ANN) classifier.

In [ ]:
# See learning curve and validation curve

def build(input_shape=(224, 224, 3), lr=1e-3, num_classes=2,
          init='normal', activ='relu', optim='adam'):
    model = Sequential()
    model.add(Conv2D(64, kernel_size=(3, 3), padding='same', input_shape=input_shape,
                     activation=activ, kernel_initializer='glorot_uniform'))
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Conv2D(64, kernel_size=(3, 3), padding='same', 
                     activation=activ, kernel_initializer='glorot_uniform'))
    model.add(MaxPool2D(pool_size=(2, 2)))
    model.add(Dropout(0.25))

    model.add(Flatten())
    model.add(Dense(128, activation='relu', kernel_initializer=init))
    model.add(Dense(num_classes, activation='softmax'))
    model.summary()

    if optim == 'rmsprop':
        optimizer = RMSprop(learning_rate=lr)
    else:
        optimizer = Adam(learning_rate=lr)

    model.compile(optimizer=optimizer, loss="binary_crossentropy", metrics=["accuracy"])
    return model

learning_rate_reduction = ReduceLROnPlateau(monitor='val_accuracy', 
                                            patience=5, 
                                            verbose=1, 
                                            factor=0.5, 
                                            min_lr=1e-7)


In [ ]:
input_shape = (224,224,3)
lr = 1e-5
init = 'normal'
activ = 'relu'
optim = 'adam'
epochs = 50
batch_size = 64

model = build(lr=lr, init= init, activ= activ, optim=optim, input_shape= input_shape)

history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs= epochs, batch_size= batch_size, verbose=0, 
                    callbacks=[learning_rate_reduction]
                   )
                   
# list all data in history
print(history.history.keys())
# summarize history for accuracy
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
K.clear_session()
del model
del history

# Step 6: Cross-Validating Model


In [ ]:
# define 3-fold cross validation test harness
kfold = KFold(n_splits=3, shuffle=True, random_state=11)

cvscores = []
for train, test in kfold.split(X_train, y_train):
  # create model
    model = build(lr=lr, 
                  init= init, 
                  activ= activ, 
                  optim=optim, 
                  input_shape= input_shape)
    
    # Fit the model
    model.fit(X_train[train], y_train[train], epochs=epochs, batch_size=batch_size, verbose=0)
    # evaluate the model
    scores = model.evaluate(X_train[test], y_train[test], verbose=0)
    print("%s: %.2f%%" % (model.metrics_names[1], scores[1]*100))
    cvscores.append(scores[1] * 100)
    K.clear_session()
    del model
    
print("%.2f%% (+/- %.2f%%)" % (np.mean(cvscores), np.std(cvscores)))

# Step 7: Testing the model

First the model has to be fitted with all the data, such that no data is left out.

In [ ]:
# Fitting model to all data
model = build(lr=lr, 
              init=init, 
              activ=activ, 
              optim=optim, 
              input_shape=input_shape)

model.fit(X_train, y_train,
          epochs=epochs, batch_size=batch_size, verbose=1,
          callbacks=[learning_rate_reduction]
         )

# Testing model on test data to evaluate
y_pred = np.argmax(model.predict(X_test), axis=1)
acc = accuracy_score(np.argmax(y_test, axis=1), y_pred)
print(f'Test Accuracy: {acc * 100:.2f}%')


In [ ]:
# save model
# serialize model to JSON
model_json = model.to_json()

with open("model.json", "w") as json_file:
    json_file.write(model_json)
    
# serialize weights to HDF5
model.save_weights("model.h5")
print("Saved model to disk")

# Clear memory, because of memory overload
del model
K.clear_session()

# Step 8: ResNet50
The CNN above is not a very sophisticated model, thus the resnet50, is also tried

In [ ]:
input_shape = (224, 224, 3)
lr = 1e-5
epochs = 50
batch_size = 64

model = ResNet50(include_top=True,
                 weights=None,
                 input_tensor=None,
                 input_shape=input_shape,
                 pooling='avg',
                 classes=2)

model.compile(optimizer=Adam(learning_rate=lr),
              loss="binary_crossentropy", 
              metrics=["accuracy"])

history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs=epochs, batch_size=batch_size, verbose=1, 
                    callbacks=[learning_rate_reduction]
                   )

# summarize history for accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.title('ResNet50 Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()

# summarize history for loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.title('ResNet50 Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Train ResNet50 on all the data
model.fit(X_train, y_train,
          epochs=epochs, batch_size= epochs, verbose=0,
          callbacks=[learning_rate_reduction]
         )


In [ ]:
# ====================================================================
# 💾 EVALUATION & ARTIFACT PACKAGING (Skin Cancer)
# ====================================================================
import os
import json
import zipfile
import shutil
from sklearn.metrics import classification_report, confusion_matrix
from IPython.display import FileLink, display

# 1. Evaluation
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
print(f'🎯 Final ResNet50 Test Accuracy: {acc * 100:.2f}%')

classes_dict = {0: 'Benign', 1: 'Malignant'}
print('
' + classification_report(y_true, y_pred, target_names=['Benign', 'Malignant']))

# 2. Output directories setup
os.makedirs('../models', exist_ok=True)
saved_files = []

# 3. Save Models (Keras modern native & H5)
keras_path = 'skin_cancer_model.keras'
h5_path = 'skin_cancer_model.h5'
model.save(keras_path)
model.save(h5_path)
saved_files.extend([keras_path, h5_path])
print(f'✓ Saved complete models: {keras_path} and {h5_path}')

# Also save JSON architecture & weights
with open('resnet50.json', 'w') as f:
    f.write(model.to_json())
model.save_weights('resnet50.h5')
saved_files.extend(['resnet50.json', 'resnet50.h5'])

# 4. Save Class Labels
classes_file = 'classes.json'
with open(classes_file, 'w') as f:
    json.dump(classes_dict, f, indent=2)
saved_files.append(classes_file)
print(f'✓ Saved class labels: {classes_file}')

# 5. Save Performance Metrics
metrics_file = 'metrics.json'
metrics_data = {
    'accuracy': float(acc),
    'architecture': 'ResNet50',
    'classes': classes_dict,
    'input_shape': [224, 224, 3]
}
with open(metrics_file, 'w') as f:
    json.dump(metrics_data, f, indent=2)
saved_files.append(metrics_file)

# Copy to repo models directory if available
for f in saved_files:
    try:
        shutil.copy(f, os.path.join('../models', f))
    except Exception:
        pass

# 6. Package all artifacts into skin_cancer_artifacts.zip
zip_filename = 'skin_cancer_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_name in set(saved_files):
        if os.path.exists(file_name):
            zipf.write(file_name, os.path.basename(file_name))
            print(f'📦 Added to archive: {file_name} ({os.path.getsize(file_name):,} bytes)')

try:
    shutil.copy(zip_filename, os.path.join('../models', zip_filename))
except Exception:
    pass

print('')
print('=' * 60)
print(f'✅ All artifacts successfully saved and packaged into: {zip_filename}')
print(f'📊 Archive Size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB')
print('=' * 60)
print('')
print('👉 Click the link below to download your complete model bundle:')
display(FileLink(zip_filename))

print('
Individual file downloads:')
for file_name in set(saved_files):
    if os.path.exists(file_name):
        display(FileLink(file_name))
